In [3]:
import pandas as pd
import sqlite3
from pathlib import Path
from sqlalchemy import create_engine, text

In [4]:
# Use pathlib to construct paths relative to the project root

BASE_DIR = Path.cwd().parent #-- this assumes the project root is one level up from the notabooks directory(outside of notebooks folder)
# if no .parent ,then it will be the current working directory(notebooks folder)assume database folder is inside notebooks folder.

DB_PATH  = BASE_DIR / 'database' / 'olist.db'
SCH_PATH = BASE_DIR / 'database' / 'schema.sql'
RAW_PATH = BASE_DIR / 'data' / 'raw'

print("Running from:", BASE_DIR)
print("DB path      :", DB_PATH)
print("SCH path     :", SCH_PATH)
print("Raw path     :", RAW_PATH)

# Create SQLAlchemy engine for SQLite
engine = create_engine(f'sqlite:///{DB_PATH}')


Running from: c:\Github Project\olist-ecommerce-analysis
DB path      : c:\Github Project\olist-ecommerce-analysis\database\olist.db
SCH path     : c:\Github Project\olist-ecommerce-analysis\database\schema.sql
Raw path     : c:\Github Project\olist-ecommerce-analysis\data\raw


In [5]:
# Create a connecttion to SQLite database and execute the schema script to crete the table
# help create the database
conn = sqlite3.connect(str(DB_PATH))

with open(SCH_PATH, 'r') as f:
    schema = f.read()

conn.executescript(schema)
conn.close()

In [6]:
# Clear existing data from tables before Loading new data 
# This is important to avoid duplicate data if notebook is run multiple time to check update or fix code or run again 

tables_to_clear = [
    'geolocation',
    'order_reviews',
    'order_payments',
    'order_items',
    'orders',
    'sellers',
    'products',
    'customers'
]

with engine.connect() as conn:
    for table in tables_to_clear:
        conn.execute(text(f"DELETE FROM {table}"))
    conn.commit()

In [7]:
# Load CSV files into database tables use append mode to add data if table exits
files = {
    'customers'      : 'olist_customers_dataset.csv',
    'products'       : 'olist_products_dataset.csv',
    'sellers'        : 'olist_sellers_dataset.csv',
    'geolocation'    : 'olist_geolocation_dataset.csv',
    'orders'         : 'olist_orders_dataset.csv',
    'order_items'    : 'olist_order_items_dataset.csv',
    'order_payments' : 'olist_order_payments_dataset.csv',
    'order_reviews'  : 'olist_order_reviews_dataset.csv',
}

for table_name, filename in files.items():
    df = pd.read_csv(RAW_PATH / filename)

    # order_reviews CSV has duplicate review_ids — drop them before inserting 
    # Detect Intergrity error by PRIMARY KEY to check duplicate review_id and drop them to make sure review_id is unique
    # keep='first' means keep the first occurrence, discard the rest

    if table_name == 'order_reviews':
        before = len(df)
        df = df.drop_duplicates(subset='review_id', keep='first')
        print(f"order_reviews: dropped {before - len(df):,} duplicate review_ids")

    df.to_sql(table_name, engine, if_exists='append', index=False)
    print(f"{table_name} — {len(df):,} rows")

customers — 99,441 rows
products — 32,951 rows
sellers — 3,095 rows
geolocation — 1,000,163 rows
orders — 99,441 rows
order_items — 112,650 rows
order_payments — 103,886 rows
order_reviews: dropped 814 duplicate review_ids
order_reviews — 98,410 rows


In [8]:
# After Loading data,run a simple query to check the numeber of rows in each table to verify data is loaded correctly
# this cell help detect the duplicate data in cell 4 with intergrity error by duplicate in Review_id and help me know data raw load half only
query = """
    SELECT 'customers'      AS tbl, COUNT(*) AS rows FROM customers UNION ALL
    SELECT 'orders',                COUNT(*)          FROM orders UNION ALL
    SELECT 'order_items',           COUNT(*)          FROM order_items UNION ALL
    SELECT 'order_payments',        COUNT(*)          FROM order_payments UNION ALL
    SELECT 'order_reviews',         COUNT(*)          FROM order_reviews UNION ALL
    SELECT 'products',              COUNT(*)          FROM products UNION ALL
    SELECT 'sellers',               COUNT(*)          FROM sellers UNION ALL
    SELECT 'geolocation',           COUNT(*)          FROM geolocation
"""

pd.read_sql(query, engine)

,tbl,rows
0,customers,99441
1,orders,99441
2,order_items,112650
3,order_payments,103886
4,order_reviews,98410
5,products,32951
6,sellers,3095
7,geolocation,1000163
